# Monitoring-Event Review

Roadmap **Phase 4** review surface (ADR-0099). This notebook **only imports
existing `autogis.core` behavior and displays it** — no business rules live in
these cells. It runs headlessly (no ArcGIS Pro) against the sanitized reference
event under `tests/fixtures/reference_event/`.

**Restart & Run All** reproduces the full review. Sections: provenance, import
summary, QA / completeness / screening / comparisons / trends (via the event
report), map-ready data, readiness, and reviewer decision.

In [ ]:
# --- Provenance: locate the repo, pin the code path, hash the inputs ---
import sys, pathlib, re

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
# Force this checkout ahead of any stale editable install of autogis.
sys.path.insert(0, str(REPO_ROOT))
FIXTURES = REPO_ROOT / "tests" / "fixtures" / "reference_event"

try:
    from importlib.metadata import version, PackageNotFoundError
    try:
        AUTOGIS_VERSION = version("autogis")
    except PackageNotFoundError:
        AUTOGIS_VERSION = None
except Exception:
    AUTOGIS_VERSION = None
if AUTOGIS_VERSION is None:  # uninstalled checkout: read pyproject
    _m = re.search(r'^version\s*=\s*"([^"]+)"',
                   (REPO_ROOT / "pyproject.toml").read_text(encoding="utf-8"), re.M)
    AUTOGIS_VERSION = _m.group(1) if _m else "unknown"

import autogis
from autogis.core.envmon.source_registry import compute_sha256

print(f"AutoGIS version : {AUTOGIS_VERSION}")
print(f"Code path       : {pathlib.Path(autogis.__file__).parent}")
print(f"Reference event : {FIXTURES}")
print("Input SHA-256 (first 16 hex):")
for _name in ["results.csv", "screening.yaml", "schedule.yaml", "samples.csv",
              "wells.csv", "reviewer_tracker.csv"]:
    print(f"  {_name:22s} {compute_sha256(FIXTURES / _name)[:16]}")

In [ ]:
# --- Run the headless producers (each is existing CLI/core behavior) ---
import os, subprocess, tempfile
WORK = pathlib.Path(tempfile.mkdtemp(prefix="event_review_"))
# Redirect the CLI run-recorder into WORK so this review never appends synthetic
# runs to a real run_history.csv (the recorder defaults to <cwd>/run_history.csv).
_ENV = {**os.environ, "AUTOGIS_RUN_HISTORY": str(WORK / "run_history.csv")}

def _run(*args):
    # cwd=REPO_ROOT so `-m autogis` resolves this checkout over the installed one.
    subprocess.run([sys.executable, "-m", "autogis", "envmon", *args],
                   cwd=str(REPO_ROOT), env=_ENV, check=True,
                   capture_output=True, text=True)

_run("apply-screening", "--results-csv", str(FIXTURES / "results.csv"),
     "--screening", str(FIXTURES / "screening.yaml"),
     "--output", str(WORK / "screened.csv"), "--site", "H281", "--event", "2026Q2")
_run("compare-events", "--results-csv", str(WORK / "screened.csv"),
     "--output", str(WORK / "comparison.csv"), "--current-event-date", "2026-06-15")
_run("run-history-report", "--results-csv", str(WORK / "screened.csv"),
     "--output", str(WORK / "history.csv"))
_run("compare-schedule-vs-actual", "--schedule", str(FIXTURES / "schedule.yaml"),
     "--results-csv", str(WORK / "screened.csv"),
     "--output", str(WORK / "gaps.csv"), "--event-date", "2026-06-15")
_run("evaluate-rpd-qa", "--samples-csv", str(FIXTURES / "samples.csv"),
     "--results-csv", str(WORK / "screened.csv"), "--report", str(WORK / "rpd_qa.csv"))
print("Producer outputs written to", WORK)

In [ ]:
# --- Import summary ---
import csv
from IPython.display import display, Markdown

_rows = list(csv.DictReader((WORK / "screened.csv").open(encoding="utf-8-sig")))
_samples = {r["SampleID"] for r in _rows}
_wells = {r["LocationID"] for r in _rows}
_analytes = {r["AnalyteCanonicalName"] for r in _rows}
_exc = [r for r in _rows if str(r.get("ExceedsScreeningLevel", "")) in ("1", "True", "true")]
display(Markdown(
    f"| Metric | Count |\n|---|---|\n"
    f"| Analytical results | {len(_rows)} |\n"
    f"| Samples | {len(_samples)} |\n"
    f"| Wells | {len(_wells)} |\n"
    f"| Analytes | {len(_analytes)} |\n"
    f"| Screening exceedances | {len(_exc)} |"))

In [ ]:
# --- QA / completeness / screening / comparisons / trends (event report HTML) ---
from datetime import date
from autogis.core.common.qa import QACollector
from autogis.core.envmon.generate_event_report import generate_event_report_html
from IPython.display import HTML, display

_report_html = generate_event_report_html(
    "H281", "2026Q2",
    results_csv=WORK / "screened.csv",
    comparison_csv=WORK / "comparison.csv",
    history_csv=WORK / "history.csv",
    gaps_csv=WORK / "gaps.csv",
    rpd_qa_csv=WORK / "rpd_qa.csv",
    generated_date=date(2026, 6, 20),
    qa=QACollector(),
)
display(HTML(_report_html))

In [ ]:
# --- Map-ready data (GeoJSON feature collection) ---
import json
from autogis.core.envmon.export_geojson import build_geojson, load_well_coords
from autogis.core.common.records_csv import read_records_csv
from autogis.core.envmon.gdb_schema import AnalyticalResultRecord

_records = read_records_csv(WORK / "screened.csv", AnalyticalResultRecord)
_coords = load_well_coords(FIXTURES / "wells.csv")
_fc = build_geojson(_records, _coords, qa=QACollector())
print(f"GeoJSON FeatureCollection: {len(_fc['features'])} feature(s)")
print(json.dumps(_fc["features"][0], indent=2))

In [ ]:
# --- Readiness state + reviewer decision ---
from autogis.core.common.run_history import RunHistory
from autogis.core.envmon.evaluate_readiness import evaluate_readiness
from autogis.core.envmon.ingest_reviewer_comments import (
    read_tracker_csv, format_comment_summary)

# Readiness reflects THIS review's actual producer runs (WORK/run_history.csv,
# recorded in the pipeline cell above), not a canned fixture. Only producers that
# record a site identity can be gated; of the headless CSV tools this review runs,
# apply-screening carries --site. A full pre-delivery gate also requires the LOCAL
# event-production tools (import -> figures), which this headless review does not run.
_readiness = evaluate_readiness(
    "H281", "2026Q2", RunHistory(WORK / "run_history.csv"), ["apply-screening"])
print(f"Readiness (this run's site-tagged producers): {_readiness.status()}")
for _r in _readiness.records:
    print(f"  [{_r.severity}] {_r.category}: {_r.message}")

print()
print(format_comment_summary(read_tracker_csv(FIXTURES / "reviewer_tracker.csv")))